# 3.3 Retrieval: similarity, MMR, hybrid, reranking

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Your API key

Some cells in this notebook call a model and need your own OpenAI API key. Paste it when asked. It is stored only in this notebook session, never saved to the file.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Learning objectives

- Explain cosine similarity search and name its three failure modes (near-duplicates, the lost middle, vocabulary mismatch).
- Configure MMR deliberately: what `lambda_mult` and `fetch_k` do, and when diversity beats raw relevance.
- Use `similarity_score_threshold` as an abstention hook, and say why thresholds must be recalibrated per embedding model.
- Build a hybrid dense + BM25 retriever with reciprocal rank fusion, and explain why `EnsembleRetriever` is not the answer in the 1.x era.
- Rerank candidates with `CohereRerank`, and decide when reranking pays versus when more context wins.
- Tune `k` with a small set of probe queries instead of guessing.

## The setup for this lesson

Everything in this lesson runs against a small local FAISS index. The setup matches the stack from lessons 3.1 and 3.2: OpenAI embeddings, FAISS, and the chunks you built there. If you skipped ahead, the rebuild below is five lines.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*langchain-community.*sunset.*")
# langchain-community is in sunset (no longer actively maintained) as of 2026.
# FAISS still lives there; the filter above keeps the import quiet.
# The standalone langchain-faiss package on PyPI is currently an empty stub,
# so community remains the working path for this course.

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)  # chunks from lesson 3.2

EU note before we start: this index runs entirely on your machine. Nothing leaves it during retrieval. That changes the moment you call a hosted reranker or hosted embeddings, which we will mark explicitly when we get there.

Install what is new in this lesson: `pip install langchain-cohere==0.6.0 rank-bm25==0.2.2`.

## Similarity search: the default and its honest limits

`as_retriever()` with no arguments does cosine similarity search: embed the query, take the `k` nearest chunk vectors, done. It is the baseline every other method in this lesson gets compared against.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
docs = retriever.invoke("How long does shipping take in the EU?")

It fails in three predictable ways. Learn to recognise them, because every retrieval upgrade below is a fix for one of them.

1. **Near-duplicates.** Chunks that say the same thing in similar words all sit near the query vector. Your top 6 can be five copies of the same fact and one useful thing. Embeddings reward paraphrase, not coverage.
2. **The lost middle.** Even after you fix retrieval, a long context list buries the crucial chunk in the middle of the prompt, where models underweight it (Liu et al., 2023). Retrieval quality and prompt position are two different problems; this lesson fixes the first, lesson 3.4 handles the second.
3. **Vocabulary mismatch.** Embeddings are decent at paraphrase but weak at exact tokens: product codes, error codes, names, version numbers. A query for `ERR-4471` will lose to a chunk that talks about "an error" at length. This is the one similarity search structurally cannot fix, and it is why hybrid search exists.

## MMR: relevance versus diversity

Maximal Marginal Relevance changes the selection rule. After picking the best chunk, it prefers chunks that are relevant to the query AND different from what is already picked. The balance is one number, `lambda_mult`: 1.0 means pure relevance (same as similarity), 0.0 means maximum diversity, 0.5 is the usual starting point.

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,            # how many chunks you finally get
        "fetch_k": 30,     # candidate pool MMR selects from: must be much larger than k
        "lambda_mult": 0.5,
    },
)

Two details that bite people. First, `fetch_k` is load-bearing: MMR can only diversify within the candidate pool. If `fetch_k` is 6 and `k` is 6, MMR is decoration. Fetch 20 to 50, return 5 to 8. Second, the selection loop runs on your machine, so MMR adds small CPU cost and no API cost. It is the cheapest upgrade in this lesson.

When does diversity beat raw relevance? When the answer needs several angles: comparison questions, "what are the options" questions, troubleshooting with multiple possible causes. When the query has one crisp answer ("what is the refund window?"), pure similarity is fine and MMR just rearranges.

## Score thresholds: the abstention hook

`similarity_score_threshold` returns chunks only if their similarity beats your threshold. Set `k` generously, because you will usually get fewer than `k` back.

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 10, "score_threshold": 0.6},
)
docs = retriever.invoke("How do I reset my password?")

This is your abstention hook. If the retriever returns nothing, the corpus has no good answer, and lesson 3.4 teaches the chain to say "I don't know" in exactly that case. A fixed `k` always returns something, even when nothing is relevant; that is how weak context contaminates answers.

The trap: scores are not comparable across setups. A threshold that works with `text-embedding-3-small` will not transfer to another embedding model or another vector store, because distance scales differ. Recalibrate the threshold whenever you change the embedding model or the store, using the probe queries from the end of this lesson. Never treat a threshold as a universal constant.

## Hybrid search: dense plus BM25

BM25 is keyword search with good ranking: it scores chunks by term frequency and rarity, so exact tokens win. It lives in `langchain_community.retrievers` and needs `rank-bm25` installed. It runs locally, costs nothing per query, and covers exactly the vocabulary-mismatch hole that embeddings leave.

In [ ]:
from langchain_community.retrievers import BM25Retriever  # pip install rank-bm25

bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 20   # fetch a wide candidate pool for fusion

The combining step: in the 1.x era there is no supported generic ensemble retriever. `EnsembleRetriever` moved to `langchain-classic`, the maintenance package, along with `ContextualCompressionRetriever`. Do not build new code on it. The supported pattern is a small custom retriever that fuses the two ranked lists with reciprocal rank fusion (RRF). RRF needs no score normalisation, which is the point: dense scores and BM25 scores live on different scales, and RRF sidesteps the problem by using ranks only.

In [ ]:
from collections import defaultdict
from langchain_core.retrievers import BaseRetriever

class HybridRRFRetriever(BaseRetriever):
    """Fuse a dense retriever and a BM25 retriever with reciprocal rank fusion.

    Each document scores 1 / (rrf_k + rank) per list; scores are summed.
    A document ranked #1 in both lists beats one ranked #1 in a single list.
    """
    dense: BaseRetriever
    sparse: BaseRetriever
    k: int = 6
    rrf_k: int = 60  # dampens the influence of very deep ranks; 60 is standard

    def _get_relevant_documents(self, query: str, *, run_manager=None):
        scores = defaultdict(float)
        by_id: dict = {}
        for results in (self.dense.invoke(query), self.sparse.invoke(query)):
            for rank, doc in enumerate(results):
                key = doc.metadata.get("source", "") + "|" + doc.page_content[:64]
                scores[key] += 1.0 / (self.rrf_k + rank + 1)
                by_id[key] = doc
        ranked = sorted(by_id, key=lambda key: scores[key], reverse=True)
        return [by_id[key] for key in ranked[: self.k]]

dense = vectorstore.as_retriever(search_kwargs={"k": 20})
retriever = HybridRRFRetriever(dense=dense, sparse=bm25, k=6)

When is hybrid worth it? Keyword-heavy corpora: documentation with function names and error codes, legal text with defined terms, product catalogues with SKUs. On prose-heavy corpora (essays, support articles written in full sentences), dense alone is usually enough and hybrid adds little. Measure, do not assume: the probe set at the end of this lesson works for this comparison too.

Two honest alternatives. Some vector stores ship hybrid natively (Weaviate and Pinecone have hybrid search retrievers in `langchain_community`), which is simpler if your store is one of them. And if your whole corpus fits in a few thousand chunks, plain BM25 alone is often embarrassingly competitive: fast, free, and explainable.

## Reranking: a cross-encoder reads the finalists

Retrieval is two-stage in every serious system. Stage one (embeddings, BM25) is fast and imprecise: it finds 50 to 200 candidates cheaply. Stage two is a cross-encoder reranker that reads the query together with each candidate and scores relevance properly. Cross-encoders cannot be indexed, which is why they run at query time on a shortlist, never on the whole corpus.

`CohereRerank` is the hosted default in 2026. Two things the docs bury: it requires a `COHERE_API_KEY`, and the `model` parameter is mandatory (there is no default; the constructor raises without it).

In [ ]:
from langchain_cohere import CohereRerank  # pip install langchain-cohere, needs COHERE_API_KEY

reranker = CohereRerank(model="rerank-v3.5", top_n=6)
# Non-English corpora: consider the multilingual variant instead.
# Model names change; check Cohere's docs for the current identifier.

candidates = retriever.invoke("How long does shipping take in the EU?")
reranked = reranker.compress_documents(candidates, "How long does shipping take in the EU?")

Note the call shape: `compress_documents(documents, query)`, the document-compressor interface, used directly. The old `ContextualCompressionRetriever` wrapper that used to host this step is legacy in `langchain-classic`; call the compressor directly, or wrap it in your own retriever if you want it inside a chain.

When does reranking pay? Published benchmarks in 2026 put cross-encoder reranking at roughly +10 to 15 points of recall gain for about $0.001 to $0.002 per query and 50 to 220 ms of latency. That is cheap for the gain, which is why it is the default recommendation for customer-facing RAG. It pays most when queries are long or nuanced, where embedding similarity is weakest.

When does more context win instead? When your context window is large and the question is simple, stuffing 20 chunks often works without a reranker. But remember the lost middle: more chunks is not more understanding past a point. Three to five highly relevant chunks routinely beat twenty loose ones. And self-hosted rerankers (BGE reranker v2 runs on your own GPU) remove the API dependency if EU data residency rules it out: Cohere is a Canadian company, and every candidate's text crosses the wire to them.

## Tuning k with probe queries

Do not tune retrieval by staring at answers. Write 10 to 20 real questions your users would ask, and for each one, write down which chunk(s) contain the answer. That is your probe set. Then, for each configuration (similarity, MMR, hybrid, hybrid + rerank), record whether the gold chunk appears in the top k. That is recall@k, computed by hand.

In [ ]:
probes = [
    ("How long does shipping take in the EU?", {"shipping-eu-chunk-id"}),
    ("What is the refund window?", {"refund-policy-chunk-id"}),
    # ... 10 to 20 of these, written from real user questions
]

def recall_at_k(retriever, probes, k):
    hits = 0
    for question, gold_ids in probes:
        got = {d.metadata.get("chunk_id") for d in retriever.invoke(question)[:k]}
        if got & gold_ids:
            hits += 1
    return hits / len(probes)

This needs `chunk_id` in your chunk metadata, which lesson 3.2 should have added; if not, add it now. Run this whenever you change the embedding model, the chunk size, or the threshold. Ten minutes of probe maintenance catches the regressions that users would otherwise report a month later.

## What can go wrong

- **Near-duplicate flooding.** Top-k is all paraphrases of one fact. Fix: MMR with a real `fetch_k` (20 to 50), not a bigger `k`.
- **`fetch_k` too close to `k`.** MMR with a candidate pool barely larger than the output is similarity search with extra steps. Keep the pool at least 3 to 5 times `k`.
- **Threshold copied from a tutorial.** Similarity scores differ per embedding model and vector store. A `score_threshold` of 0.7 from someone else's stack means nothing in yours. Recalibrate with probes whenever the embedding model changes.
- **BM25 silently degrades on unclean text.** The default preprocessor lowercases and strips punctuation, which is right for prose but can mangle code and product SKUs. If keyword retrieval underperforms, check `preprocess_func` before blaming BM25.
- **Reranking the wrong candidates.** Reranking 10 candidates gives the reranker no headroom; reranking 1000 melts latency. Retrieve 50 to 200, rerank down to your `k`.
- **Legacy imports.** `EnsembleRetriever` and `ContextualCompressionRetriever` import fine from `langchain_classic`, and tutorials still recommend them. They are maintenance-mode code. New code uses a custom `BaseRetriever` for fusion and calls the compressor directly.
- **The community sunset warning.** Importing anything from `langchain_community` prints a sunset deprecation warning. It is cosmetic for now; keep FAISS there, filter the warning in lesson code, and watch the standalone integration packages.
- **Every hosted step is a data transfer.** Hosted embeddings and Cohere rerank send your documents' text to another company. For EU deployments under GDPR, that needs a legal basis and usually a data processing agreement. The local path (self-hosted embeddings, BM25, self-hosted reranker, local FAISS) keeps everything on your infrastructure.

## Check your understanding

**1. Name the three failure modes of pure similarity search, and say which one only hybrid search fixes.**
Near-duplicates (several paraphrases of one fact crowd out coverage), the lost middle (the key chunk is retrieved but buried mid-prompt where the model underweights it), and vocabulary mismatch (exact tokens like error codes and product names lose to long paraphrases). Only hybrid search fixes vocabulary mismatch, because it adds a keyword scorer that rewards exact terms; MMR and reranking address the other two.

**2. You set `search_type="mmr"` with `k=6, fetch_k=8, lambda_mult=0.5` and see no improvement over similarity. What is wrong, and what do you change?**
The candidate pool is too small: with `fetch_k=8` and `k=6`, MMR can barely diversify. Raise `fetch_k` to 30 or so, keeping `k=6`. Then, if results are still too samey, lower `lambda_mult` toward 0.3 to weight diversity more; if relevance drops, raise it toward 0.7.

**3. Why is `EnsembleRetriever` the wrong choice for a new hybrid retriever in the 1.x era, and what do you build instead?**
It lives in `langchain-classic`, the maintenance package, and receives no new features; new code should not be built on it. Instead, subclass `langchain_core.retrievers.BaseRetriever` and fuse the dense and BM25 ranked lists yourself, for example with reciprocal rank fusion, which avoids normalising scores across the two different scales.

**4. A reranker is quoted at 150 ms and $0.0015 per query. Your app serves 200,000 queries a month. Work out the monthly rerank cost and name one situation where you would drop the reranker anyway.**
200,000 times $0.0015 is $300 per month, plus the latency on every query. You would drop it when the corpus and questions are simple enough that 3 to 5 well-retrieved chunks answer reliably without reranking, or when EU data residency rules forbid sending document text to a third-party API and no self-hosted reranker is available.

**5. After switching embedding models, your `similarity_score_threshold` retriever suddenly returns almost nothing. What happened, and what is the fix?**
Similarity score scales differ between embedding models, so the old `score_threshold` no longer matches the new score distribution; chunks that are relevant now score below the cutoff. Recalibrate the threshold against your probe set on the new model, and record the new value in VERSIONS.md with the model name.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.